# 04 · Timing, BatchNorm check, baseline  (steps 8-9)
**Accelerator: GPU T4 x2.** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


1. **Timing** (5 min): estimated hours per full run → plan the rest from it (uploaded to `timing/`).
2. **Queue `queues/checks.yaml`:** BatchNorm-check runs (micro-batch 1 / 4 / 4-with-stock-BN) and the
   baseline seed 0. Both T4s train different jobs. Re-run until `status` shows all four `done`.
Scores for the checks are computed by notebook 06 (CPU).

In [ ]:
# ==== EDIT ONLY THIS CELL (keep the same values on every Kaggle account) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # private Hugging Face dataset repo; created automatically
ACCOUNT = "account-1"                         # any label; recorded in progress.json (who trained what)
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Kaggle setup (once per account): Add-ons -> Secrets -> add HF_TOKEN (a WRITE token) and tick it for this
# notebook; Settings -> Internet ON; Settings -> Accelerator as noted in the title.

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub>=0.25", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
if not os.path.exists(REPO):
    attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
    if attached:                                   # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
        CODE_SOURCE = "kaggle dataset"
    elif REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
        CODE_SOURCE = "git"
    else:                                          # copy pushed to Hugging Face by notebook 01
        from huggingface_hub import snapshot_download
        snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                          token=os.environ["HF_TOKEN"])
        shutil.copytree(f"{W}/hf_code/code", REPO)
        CODE_SOURCE = "huggingface"
    print("code from:", CODE_SOURCE)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py

In [ ]:
from pedger_kd.sync import open_remote
remote = open_remote(os.environ["PEDGER_REMOTE"])
remote.download("data/bsds", WS)
if not remote.exists("timing/baseline.json"):
    !python bench/time_run.py --config configs/baseline.yaml --data-root {WS}/data --out {W}/timing/baseline.json
    !python -m pedger_kd.sync push-dir --local {W}/timing --to timing
else:
    remote.download("timing", W)
!cat {W}/timing/baseline.json

In [ ]:
HOURS = 11.25            # whole-session budget (Kaggle stops GPU sessions at 12 h)
!python -m pedger_kd.jobs --queue queues/checks.yaml --ws {WS} --hours {HOURS} --session-start {SESSION_START} --gpus auto --label {ACCOUNT}

In [ ]:
!python -m pedger_kd.sync status

In [ ]:
# keep the notebook output small (everything important is already on Hugging Face)
for d in ["ws/data", "ws/teachers", "ws/.staging", "hf_code", "work", "BSR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h")